# Universidad Central · Maestría en Analítica de Datos
# S10 · De imágenes y audio a evidencia multimodal

**Problema de Laura.** Después de organizar noticias y documentos (S03–S08) y recuperar textos (S09), ¿cómo prepara, mide, compara y consulta **imágenes y audio** sin confundir propiedades técnicas con significado?

**Dos recursos para esta clase:** [Presentación-aplicación y guía integrada](https://jazaineam1.github.io/BigData2026/Presentaciones/s10-imagenes-audio.html#s1) y **este cuaderno**. No requiere cuentas de pago, credenciales de terceros ni descarga de audios periodísticos.

**Qué vas a producir:** originales propios del laboratorio, dos tamaños de imagen, una señal WAV de referencia y **10 grabaciones reales de voz en español con licencia CC BY 4.0**, histogramas, formas de onda y espectrograma, características tabulares, catálogo JSONL/Parquet, manifiesto y decisión individual cerrada.

**Mapa sin relojes:** iniciar Colab → imagen: medir y cambiar tamaño → audio: muestras y ventanas → descargar 10 voces reales → escuchar y medir → (opcional) ASR → distinguir inventarios de noticias y voces → catálogo → consulta → decisión → autocomprobación → exportación.

**Derechos y límites:** las direcciones de audio indicadas por feeds editoriales se estudian como metadatos, no se rastrean ni se descargan. El laboratorio usa una ilustración y una señal generadas localmente. Si analizas tu propia grabación, pide permiso cuando intervengan otras personas; en Colab, los archivos que subes llegan al entorno de Google.

## 1 · Colab desde cero: instrucciones para un estudiante independiente

**Antes de ejecutar:** necesitas un navegador, una cuenta gratuita de Google y conexión a internet. No instales nada en tu computador, no uses terminal, no uses VS Code y no pagues por una API. Las instrucciones detalladas están en la [presentación, paso 12](https://jazaineam1.github.io/BigData2026/Presentaciones/s10-imagenes-audio.html#s12).

**Paso 1.** Entra al cuaderno mediante el enlace de la presentación. Pulsa **Conectar** arriba a la derecha; cuando Colab asigne CPU/RAM, verás un indicador de conexión. No necesitas GPU ni un runtime especial.

**Paso 2.** En el menú **Archivo → Guardar una copia en Drive**. Se abrirá una copia editable en tu Google Drive. No compartas una carpeta que contenga información sensible.

**Paso 3.** Ejecuta las celdas **en orden**, desde esta primera sección hasta el final, usando ▶ a la izquierda de cada celda o **Entorno de ejecución → Ejecutar todas** para la primera pasada. Si el navegador solicita permiso para ejecutar el cuaderno, revisa que provenga del repositorio del curso. La primera ejecución instala dependencias gratuitas y descarga el conjunto de 10 voces autorizado.

**Paso 4.** Revisa **cada tabla, gráfico y mensaje de salida** según los cuatro rótulos «Cómo se lee / Qué nos dice / Qué NO permite concluir / Qué error común». Después modifica los selectores indicados, vuelve a ejecutar **solo la celda modificada** y compara. Nunca confundas «se ejecutó» con «se comprendió».

**Paso 5.** Completa tus tres decisiones estructuradas, ejecuta **Autocomprobación** hasta ver el estado **LISTO**, y conserva el paquete con el método **Descargar ZIP** o **Guardar copia en Drive** al final. La copia del cuaderno no conserva los archivos temporales del runtime: debes guardar también el paquete.

**Comprobación inmediata:** la primera celda debe imprimir \`BIBLIOTECAS: listas\` y una ruta terminada en \`work_s10_multimodal\`. Si aparece un error: **Entorno de ejecución → Reiniciar sesión**, y vuelve a ejecutar las celdas en orden. **No publiques una captura con tokens, correos o rutas privadas.**

**Por qué aparecen estas funciones:** \`Path.mkdir()\` crea la carpeta del paquete; \`find_spec()\` revisa las bibliotecas y \`subprocess.check_call()\` instala solo las que falten. Nunca pegues credenciales de plataformas externas en este cuaderno.

In [ ]:
#@title Preparar bibliotecas (no reemplazar NumPy/Pandas precargados)
import importlib.util, subprocess, sys, os
faltan=[]
for modulo, paquete in [('PIL','pillow'),('numpy','numpy'),('pandas','pandas'),('pyarrow','pyarrow'),('matplotlib','matplotlib'),('soundfile','soundfile>=0.12'),('IPython','ipython')]:
    if importlib.util.find_spec(modulo) is None: faltan.append(paquete)
if faltan: subprocess.check_call([sys.executable,'-m','pip','install','-q',*faltan])
print('BIBLIOTECAS:', 'listas; no se necesitan servicios externos')
from IPython.display import display, Audio
from pathlib import Path
RAIZ=Path('work_s10_multimodal').resolve()
RAW=RAIZ/'raw'; DER=RAIZ/'derived'; OUT=RAIZ/'outputs'; CHECKS=RAIZ/'checks'
for carpeta in (RAW,DER,OUT,CHECKS): carpeta.mkdir(parents=True,exist_ok=True)
print('ÁREA DE TRABAJO:',RAIZ)

## 2 · Contexto: la misma clave no es el mismo contenido

**Ejemplo de esquema proporcionado para la clase:** una noticia puede incluir \`audio_url\`; otras referencias apuntan a imágenes. **Aquí solo reconocemos y validamos campos**, no solicitamos esos archivos ni concluimos que imagen y audio contienen lo mismo.

**Cómo se lee:** \`article_id\` identifica el registro editorial, \`audio_url\` es una referencia declarada. **Qué nos dice:** cómo unir metadatos de distintas modalidades. **Qué NO permite concluir:** que el audio existe ahora, que tiene licencia para minería o que describe la foto. **Error común:** inferir disponibilidad o derechos a partir de una URL.

In [ ]:
# Ejemplo de estructura, sin consultar ningún sitio periodístico.
from urllib.parse import urlparse
feed_ejemplo={'article_id':'3530042',
             'article_feed':'https://www.eltiempo.com/servicios/feeds/articulo/3530042',
             'audio_url':'https://servicios.eltiempo.com/audios/3530042.wav'}
assert urlparse(feed_ejemplo['audio_url']).scheme=='https'
assert feed_ejemplo['audio_url'].endswith('/'+feed_ejemplo['article_id']+'.wav')
print('REFERENCIA DE AUDIO (NO DESCARGADA):',feed_ejemplo['audio_url'])
print('ACTIVOS DEL LABORATORIO: originales del curso, independientes de ese artículo')

## 3 · Imagen: de una matriz a preguntas observables

**Píxel.** Es una posición de una imagen digital. En RGB, sus tres canales rojo/verde/azul suelen estar entre 0 y 255. Ejemplo manual: \`(255,0,0)\` es rojo, \`(0,0,0)\` es negro. Una imagen de 4 × 3 píxeles contiene 12 posiciones; si es RGB de 8 bits por canal, representa 36 valores de canal antes de compresión.

**Luminancia.** Aproximación a la claridad de cada píxel: \`Y = 0.2126×R + 0.7152×G + 0.0722×B\`. Negro da 0; blanco da 255. El resultado NO describe texto, objetos o semántica.

**Histograma.** Tabla/gráfico de **conteos de píxeles por intervalos de luminancia**. Eje horizontal: de 0 oscuro a 255 claro; vertical: cuántos píxeles aparecen en cada intervalo. Si todos los píxeles fueran blancos, la barra derecha concentraría el conteo. Si agrupas en 4 intervalos hay unos 64 niveles por barra; con 16 hay 16 niveles por barra. **Cambiar el número de barras no modifica la imagen**, solo la descripción.

**Resolución.** Columnas × filas; no equivale a bytes del archivo comprimido. **Contraste.** Separa o junta los valores alrededor de la claridad media. **Brillo.** Aumenta o disminuye las intensidades. **Umbral.** Valor que separa claros y oscuros. **Binarización.** Convierte los píxeles en negro o blanco según el umbral.

**HAZ ESTO AHORA.** Abre la [presentación (experimento de imagen, diapositiva 5)](https://jazaineam1.github.io/BigData2026/Presentaciones/s10-imagenes-audio.html#s5), cambia brillo, contraste, umbral y número de barras; toca el resultado para inspeccionar un píxel. Después vuelve a este cuaderno y ejecuta las celdas de imagen en orden.

**Cómo se lee:** cada barra del histograma cuenta intensidades y cada fila de la tabla mide una representación de la imagen. **Qué nos dice:** cómo se distribuyen claridad y resolución. **Qué NO permite concluir:** que un OCR reconocería el texto. **Qué error común:** creer que menos barras significan menos píxeles.

In [ ]:
# E — imagen ORIGINAL creada por el curso (sin fuentes de terceros).
from PIL import Image,ImageDraw,ImageFont
import numpy as np, hashlib, json, shutil, wave, math
import pandas as pd, matplotlib.pyplot as plt
def sha256(ruta):
    h=hashlib.sha256()
    with open(ruta,'rb') as f:
        for bloque in iter(lambda:f.read(1024*1024),b''):h.update(bloque)
    return h.hexdigest()
imagen_original=RAW/'documento_demo.png'
if not imagen_original.exists():
    im=Image.new('RGB',(640,360),'#e8edf6');d=ImageDraw.Draw(im)
    d.rectangle((38,22,599,337),fill='white',outline='#1c365b',width=6)
    d.rectangle((70,52,310,94),fill='#23647a')
    for n in range(11):d.rectangle((72,116+n*17,405+(n%3)*42,121+n*17),fill='#657a90')
    d.rectangle((440,68,550,192),fill='#ebc46c')
    d.ellipse((460,93,528,164),fill='#4b9091')
    d.text((70,305),'BIG DATA - DOCUMENTO FICTICIO',fill='#213b52')
    im.save(imagen_original)
print('E · ORIGINAL:',imagen_original.name,sha256(imagen_original)[:16])
display(Image.open(imagen_original))

### Pregunta: ¿reducir una imagen conserva toda la información?

**HAZ ESTO AHORA.** Cambia **solo** \`LADO=320\` por \`160\` y vuelve a ejecutar. Compara proporción, bytes y legibilidad. Luego decide cuál perfil sirve como vista previa del catálogo.

**Función:** \`Image.thumbnail((lado,lado))\` reduce manteniendo proporción. No modifica el archivo RAW si trabajas con una copia y escribes un derivado distinto.

In [ ]:
# T — un solo parámetro: 160 o 320. El RAW permanece intacto.
LADO=320 #@param [160,320] {type:"raw"}
if type(LADO) is not int or LADO not in (160,320): raise ValueError('Elige 160 o 320')
hash_antes=sha256(imagen_original)
derivada=DER/f'documento_{LADO}.png'
with Image.open(imagen_original) as im:
    copia=im.copy();copia.thumbnail((LADO,LADO),Image.Resampling.LANCZOS);copia.save(derivada)
assert sha256(imagen_original)==hash_antes
with Image.open(derivada) as im:
    print('T · TAMAÑO:',im.size,'· bytes:',derivada.stat().st_size,'· RAW conservado:',True)
    display(im)

### De una imagen a números: qué observar

- **\`np.asarray(imagen)\`** devuelve los píxeles como matriz NumPy: altura × ancho × canales.
- **Luminancia media:** promedio de las intensidades aproximadas. Si una imagen se aclara, normalmente aumenta.
- **Histograma:** \`np.histogram(..., bins=...)\` suma las ocurrencias por intervalo. La suma de los conteos debe ser igual al número de píxeles usados. Cambiar \`bins\` NO cambia la imagen.
- **Diferencia vecina:** mide cambios locales entre valores contiguos; sirve para inspeccionar textura o bordes, no reconocer objetos.

**HAZ ESTO AHORA.** Ejecuta la tabla de características y abre la diapositiva 5 para comparar visualmente los mismos conceptos con el brillo y el número de intervalos editables. Observa las dos filas y no interpretes un promedio como un diagnóstico semántico.

**Cómo se lee:** compara magnitudes entre perfiles del mismo original. **Qué nos dice:** qué cambió al reducir la resolución. **Qué NO permite concluir:** precisión de OCR. **Qué error común:** interpretar el histograma como un mapa de dónde están los objetos.

In [ ]:
# Propiedades cuantificables del mismo original y los dos perfiles.
def imagen_caracteristicas(ruta):
    with Image.open(ruta) as img:
        rgb=np.asarray(img.convert('RGB'),dtype=np.float32)
    gris=.2126*rgb[:,:,0]+.7152*rgb[:,:,1]+.0722*rgb[:,:,2]
    h,w=gris.shape
    dx=np.abs(np.diff(gris,axis=1));dy=np.abs(np.diff(gris,axis=0))
    return {'width':int(w),'height':int(h),'pixels':int(w*h),
      'mean_luminance':round(float(gris.mean()),3),
      'bright_fraction':round(float((gris>=180).mean()),4),
      'horizontal_change':round(float(dx.mean()),3),
      'vertical_change':round(float(dy.mean()),3),
      'stored_bytes':int(Path(ruta).stat().st_size)}
comparacion=[]
for lado in (160,320):
    ruta=DER/f'documento_{lado}.png'
    with Image.open(imagen_original) as im:
        copia=im.copy();copia.thumbnail((lado,lado),Image.Resampling.LANCZOS);copia.save(ruta)
    comparacion.append({'perfil':f'vista{lado}',**imagen_caracteristicas(ruta)})
display(pd.DataFrame(comparacion))
grises=np.asarray(Image.open(imagen_original).convert('L'))
plt.figure(figsize=(8,2.7));plt.hist(grises.ravel(),bins=32,range=(0,256));plt.xlabel('Intensidad (0–255)');plt.ylabel('Píxeles');plt.title('Histograma del original del curso');plt.show()
print('CÓMO SE LEE: compara el cambio de resolución y de distribución; el histograma no devuelve texto OCR.')

**Cómo se lee:** cada fila corresponde a un tamaño y su brillo, sus cambios locales y bytes almacenados. **Qué nos dice:** el efecto de reducir esta ilustración particular. **Qué NO permite concluir todavía:** calidad de OCR en documentos reales; necesitaríamos un conjunto de documentos y un texto de referencia. **Qué error común:** igualar tamaño comprimido con cantidad de información visual útil.

**Fállalo aquí, que sale gratis:** ¿por qué un histograma no identifica el artículo de una foto? Porque cuenta intensidades sin conservar todas sus ubicaciones.

## 4 · Audio: muestra, onda, RMS, FFT y espectrograma

**Muestra:** un valor numérico de amplitud registrado en un instante. **Frecuencia de muestreo (Hz):** cuántas muestras se registran por segundo: 16.000 muestras/s × 3 s = 48.000 muestras por canal. No es la frecuencia del tono.

**Forma de onda:** gráfico de amplitud frente a tiempo. La línea cerca de cero significa amplitud pequeña, no ausencia demostrada de palabras.

**RMS:** raíz cuadrada del promedio de los cuadrados de las amplitudes, \`sqrt(mean(x²))\`. Ejemplo manual: para \`[1, -1, 0, 0]\`, RMS = \`sqrt(2/4)\` ≈ 0,707. Un RMS mayor NO indica por sí solo emociones ni volumen percibido.

**FFT:** transformada rápida de Fourier. **Espectro:** magnitud de las componentes de frecuencia en UNA ventana. **Pico espectral:** componente más intensa de la ventana. **Tamaño de ventana:** número de muestras usadas en el cálculo; con 16 kHz, 256 muestras abarcan 16 ms y 2.048 abarcan 128 ms. La separación nominal de frecuencias es \`muestreo/muestras_ventana\`: 62,5 Hz o 7,8125 Hz respectivamente.

**Espectrograma:** varios espectros consecutivos: tiempo en el eje X, frecuencia en Y y color para energía/magnitud relativa. Un silencio puede verse oscuro.

**Contenedor y códec:** WAV es el contenedor; PCM16 codifica muestras de 16 bits, mientras G.711 puede almacenar audio comprimido dentro de WAV. En el laboratorio real conservamos los WAV de origen y generamos copias PCM16.

**HAZ ESTO AHORA.** Experimenta en la [diapositiva 8](https://jazaineam1.github.io/BigData2026/Presentaciones/s10-imagenes-audio.html#s8): cambia frecuencia, amplitud y longitud de ventana FFT; compara la forma de onda, el espectro, el RMS y el espectrograma. Luego ejecuta aquí la señal original.

**Cómo se lee:** cada gráfica responde una pregunta diferente. **Qué nos dice:** amplitud y frecuencias medibles. **Qué NO permite concluir:** contenido verbal ni intención de la persona. **Qué error común:** interpretar 16.000 Hz de muestreo como una voz que suena a 16 kHz.

In [ ]:
# E — audio WAV PCM mono de 16 bits, creado localmente sin terceros.
SR=16000
duracion=3.0
t=np.arange(int(SR*duracion),dtype=np.float64)/SR
senal=np.zeros(t.size,dtype=np.float64)
primero=(t>=.1)&(t<1.0);segundo=(t>=2.0)&(t<2.9)
senal[primero]=.45*np.sin(2*np.pi*190*t[primero])+.09*np.sin(2*np.pi*380*t[primero])
senal[segundo]=.30*np.sin(2*np.pi*360*t[segundo])+.04*np.sin(2*np.pi*720*t[segundo])
audio_original=RAW/'senal_demo.wav'
pcm=(np.clip(senal,-1,1)*32767).astype('<i2')
with wave.open(str(audio_original),'wb') as wf:
    wf.setnchannels(1);wf.setsampwidth(2);wf.setframerate(SR);wf.writeframes(pcm.tobytes())
assert audio_original.exists()
from IPython.display import Audio,display
display(Audio(senal,rate=SR))
print('E · AUDIO:',round(len(senal)/SR,2),'s ·',SR,'Hz · canal mono · WAV PCM16 ·',audio_original.stat().st_size,'bytes')

## 5 · Escuchar, dibujar y medir (y saber lo que significa)

**RMS** (raíz del promedio de cuadrados) resume amplitud eficaz: si una ventana tiene silencio total, RMS es 0. **FFT** transforma la ventana de muestras en frecuencias. **Espectro** son magnitudes por frecuencia de esa ventana. **Espectrograma** repite el cálculo para observar cómo cambian las frecuencias con el tiempo. Ni RMS ni FFT transcriben palabras.

**HAZ ESTO AHORA.** Escucha el audio sintético, mueve \`VENTANA_SEG\` de la primera región sonora a la región central silenciosa, vuelve a ejecutar y compara RMS. Mira el gráfico y reconoce dos zonas con tonos y una zona sin energía. Abre la diapositiva 8 para variar también el tamaño de la ventana FFT en la interfaz.

**Cómo se lee:** primer tono, silencio y segundo tono deben tener medidas diferentes. **Qué nos dice:** cómo cambia la señal. **Qué NO permite concluir:** si un ser humano pronunció una palabra. **Qué error común:** pensar que un pico de FFT equivale a una letra o fonema.

**Si falla:** revisa que primero ejecutaste la celda que crea \`senal_demo.wav\`; las ventanas dependen de la variable \`senal\` y de la frecuencia \`SR\`.

In [ ]:
# T — características acústicas por ventana; el audio WAV no se sobrescribe.
VENTANA_SEG=0.2 #@param [0.2,1.3,2.2] {type:"raw"}
if float(VENTANA_SEG) not in (.2,1.3,2.2):raise ValueError('Elige una ventana propuesta')
def audio_ventana(x,sample_rate,segundos,ancho=.25):
    inicio=int(segundos*sample_rate);n=int(ancho*sample_rate)
    ventana=np.asarray(x[inicio:inicio+n],dtype=np.float64)
    if len(ventana)!=n:raise ValueError('Ventana fuera de la señal')
    rms=float(np.sqrt(np.mean(ventana**2)))
    magnitud=np.abs(np.fft.rfft(ventana*np.hanning(len(ventana))))
    frecuencias=np.fft.rfftfreq(len(ventana),d=1/sample_rate)
    pico=float(frecuencias[np.argmax(magnitud[1:])+1]) if len(magnitud)>1 else 0.
    return {'start_seconds':segundos,'duration_seconds':ancho,'rms':round(rms,5),
            'peak_frequency_hz':round(pico,1),'peak_amplitude':round(float(np.max(np.abs(ventana))),4),
            'signal':ventana,'frequencies':frecuencias,'magnitudes':magnitud}
actual=audio_ventana(senal,SR,float(VENTANA_SEG))
print('VENTANA:',{k:v for k,v in actual.items() if k not in ('signal','frequencies','magnitudes')})
plt.figure(figsize=(9,2.5));plt.plot(np.arange(len(senal))/SR,senal,lw=.7);plt.xlabel('Segundos');plt.ylabel('Amplitud relativa');plt.title('Señal original: tonos y silencio');plt.show()
plt.figure(figsize=(9,2.5));plt.plot(actual['frequencies'],actual['magnitudes']);plt.xlim(0,1000);plt.xlabel('Hz');plt.ylabel('Magnitud relativa');plt.title('Espectro de TU ventana');plt.show()

### El espectrograma: frecuencias que cambian con el tiempo

Un espectrograma hace ventanas consecutivas. En este caso **debe** haber una zona prácticamente silenciosa, además de dos regiones tonales. Las frecuencias del ejemplo son conocidas por diseño; eso nos permite comprobar nuestro cálculo.

Un espectrograma no revela automáticamente quién habla ni las palabras de una noticia.

In [ ]:
# Espectrograma: FFT por segmentos, con salto menor que el tamaño de ventana.
ancho=512;salto=160
bloques=np.stack([senal[i:i+ancho] for i in range(0,len(senal)-ancho+1,salto)])
espectros=np.abs(np.fft.rfft(bloques*np.hanning(ancho),axis=1))
frecuencias=np.fft.rfftfreq(ancho,1/SR)
instantes=(np.arange(len(bloques))*salto+ancho/2)/SR
plt.figure(figsize=(9,3))
plt.pcolormesh(instantes,frecuencias,20*np.log10(espectros.T+1e-6),shading='auto')
plt.ylim(0,1200);plt.xlabel('Tiempo (s)');plt.ylabel('Frecuencia (Hz)')
plt.title('Espectrograma del audio original; dB relativos');plt.colorbar(label='dB relativos');plt.show()
metricas_ventanas=pd.DataFrame([{k:v for k,v in audio_ventana(senal,SR,s).items() if k not in ('signal','frequencies','magnitudes')} for s in (.2,1.3,2.2)])
display(metricas_ventanas)

**Cómo se lee:** hay frecuencias distintas en las regiones sonoras y un valle de energía en silencio. **Qué nos dice:** cómo cambió esta señal conocida. **Qué NO permite concluir:** que existe una voz o un contenido semántico. **Error común:** confundir muestreo (Hz del dispositivo) con pico de tono (Hz de la señal).

Más adelante analizarás **10 WAV de voz humana reales con licencia** del conjunto MINDS-14; esta señal sin voz solo fue el punto de partida para entender la FFT.

### Si tienes un WAV propio con autorización (opcional)
Puedes sustituir la señal de ejemplo con tu voz grabada voluntariamente, o un audio con licencia de análisis. Usa WAV PCM mono para seguir el recorrido sin instalar un decodificador adicional. **No subas conversaciones privadas o sin consentimiento a Colab.** Si cambias de archivo, reinicia los cálculos; no le atribuyas al audio del curso el contenido de tu grabación.

In [ ]:
#@title Opcional: usar WAV PCM propio (DESACTIVADO para ejecución de clase)
USAR_WAV_PROPIO=False #@param {type:"boolean"}
RUTA_WAV_PROPIO='' #@param {type:"string"}
AUTORIZADO_PARA_ANALISIS=False #@param {type:"boolean"}
if USAR_WAV_PROPIO:
    if not AUTORIZADO_PARA_ANALISIS:raise ValueError('Se requiere permiso para analizar este audio')
    ruta=Path(RUTA_WAV_PROPIO)
    if not ruta.is_file() or ruta.stat().st_size>20*1024*1024:raise ValueError('WAV ausente o mayor de 20 MiB')
    with wave.open(str(ruta),'rb') as wf:
        if wf.getnchannels()!=1 or wf.getsampwidth()!=2 or wf.getcomptype()!='NONE':
            raise ValueError('Usa WAV PCM16 mono, sin compresión')
        SR_PROPIO=wf.getframerate();num=min(wf.getnframes(),SR_PROPIO*30)
        muestras=np.frombuffer(wf.readframes(num),dtype='<i2').astype(np.float32)/32768
    if SR_PROPIO<8000 or SR_PROPIO>48000:raise ValueError('Muestreo fuera del alcance')
    print('AUDIO AUTORIZADO:',len(muestras)/SR_PROPIO,'segundos; RMS:',float(np.sqrt(np.mean(muestras**2))))
else:
    print('Usando la señal original del curso; sin archivos de terceros')

## 6 · Voz y significado: por qué ASR es una etapa distinta

Reconocimiento automático del habla (**ASR**) produce una hipótesis de palabras a partir de una señal que efectivamente contenga voz. Ni una FFT ni RMS transcriben palabras.

Para evaluar un sistema de transcripción necesitas una **referencia humana**; WER cuenta sustituciones, inserciones y eliminaciones de palabras y divide por la longitud de esa referencia. Con cero palabras de referencia, WER no está definido.

**Extensión de profundización (no necesaria en esta clase):** ejecutar un modelo ASR descargado y autorizado en un entorno apropiado, comparar transcripción contra referencia, y observar fallos con ruido/nombres propios. No declares calidad del modelo con el audio sintético sin voz de este cuaderno.

In [ ]:
# Ejercicio manual de evaluación de ASR: ejemplos creados para el curso, no son transcripciones editoriales.
referencia='el contrato tiene tres anexos'.split()
hipotesis='el contrato tiene dos anexos'.split()
def distancia_palabras(a,b):
    anterior=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        actual=[i]+[0]*len(b)
        for j,y in enumerate(b,1):
            actual[j]=min(anterior[j]+1,actual[j-1]+1,anterior[j-1]+(x!=y))
        anterior=actual
    return anterior[-1]
errores=distancia_palabras(referencia,hipotesis)
wer=errores/len(referencia)
print('WER DEL EJEMPLO DIDÁCTICO:',errores,'errores /',len(referencia),'palabras =',round(wer,3))
assert errores==1
print('LÍMITE: WER mide comparación textual con referencia; no la verdad del contenido.')

## 7 · Diez grabaciones **reales de voz** en español

Acabamos de comprender una señal sencilla. Ahora analizaremos **10 grabaciones reales de personas hablando español**, del conjunto **MINDS-14 / PolyAI, configuración \`es-ES\`**, publicado como **CC BY 4.0**. Son consultas habladas sobre servicios financieros; **NO** son noticias de El Tiempo, **NO** pertenecen a los 987 artículos de la colección editorial y **NO** deben unirse falsamente a ellos.

**¿Por qué este segundo origen?** Queremos ejecutar análisis acústico y transcripción de voz real sin infringir licencias. El feed de una noticia puede declarar \`audio_url\`, pero la presencia de una URL no autoriza descargar ni hacer minería de medios editoriales. El material del curso mantiene su esquema \`article_id\` para noticias, mientras los audios de MINDS-14 mantienen su propio identificador \`source_record_id\`.

**Qué tendrás al terminar:** diez WAV de origen, sus diez derivados PCM16 en \`raw/voz_real/\`, diez huellas SHA-256, diez transcripciones de referencia, duración, RMS, cambios de signo y centroide espectral; después tendrás once filas de audio (10 reales + 1 señal de prueba) y dos filas de imagen en el catálogo.

**HAZ ESTO AHORA.** Ejecuta la próxima celda. Descarga el **Parquet de español (~39 MB) una sola vez** al entorno de Colab, sin autenticación y sin API de pago. Después selecciona de manera determinista una muestra de **diez grabaciones de intenciones diferentes, conservando el WAV de origen y decodificando el códec telefónico a PCM16**. No es una muestra aleatoria representativa. Se conservan el dataset, la licencia, el registro, los bytes y la transcripción proporcionada por el corpus.

**Si falla:** revisa conectividad de Colab y reejecuta esta celda; si el proveedor no responde se detiene con error explícito. No se sustituyen las diez voces por señales inventadas. El tiempo depende de la red.

In [ ]:
#@title 7A · Conseguir 10 grabaciones reales (MINDS-14 español, CC BY 4.0)
# Los audios son voz humana del dataset; NO descargas de sitios periodísticos.
import urllib.request, io, time, collections
ARCHIVO_FUENTE=RAIZ/'cache'/'minds14_es_ES_train.parquet'
ARCHIVO_FUENTE.parent.mkdir(parents=True,exist_ok=True)
ORIGEN_VOZ='https://huggingface.co/datasets/PolyAI/minds14/resolve/main/es-ES/train-00000-of-00001.parquet'
LICENCIA_VOZ='CC-BY-4.0'
SHA256_PARQUET='989bf2f90676a5eb5852fe785a03b1d9b97763e0febbd0fd0a3f489afd83badf'
if not ARCHIVO_FUENTE.exists():
    temporal=ARCHIVO_FUENTE.with_suffix('.parquet.part')
    print('Preparando el corpus de voz real en español (~39 MB)...')
    try:
        req=urllib.request.Request(ORIGEN_VOZ,headers={'User-Agent':'BigData-UCentral-S10-educational/1.0'})
        with urllib.request.urlopen(req,timeout=120) as respuesta, temporal.open('wb') as destino:
            while True:
                parte=respuesta.read(1024*1024)
                if not parte: break
                destino.write(parte)
                if destino.tell()>100_000_000: raise ValueError('Descarga superior al límite seguro de 100 MB')
        temporal.replace(ARCHIVO_FUENTE)
    except Exception as error:
        temporal.unlink(missing_ok=True)
        raise RuntimeError('No se pudo obtener el corpus autorizado; verifica la red de Colab y reintenta (no se usarán voces sintéticas).') from error
if sha256(ARCHIVO_FUENTE)!=SHA256_PARQUET:
    raise ValueError('El Parquet español no coincide con la versión auditada; revisa el origen antes de ejecutar')
# No se descargan los archivos wav uno a uno: ya están encapsulados en el parquet.
corpus_voz=pd.read_parquet(ARCHIVO_FUENTE,columns=['audio','path','transcription','intent_class'])
if len(corpus_voz)<10:raise ValueError('El corpus tiene menos de 10 grabaciones')
corpus_voz=corpus_voz.sort_values('path',kind='stable').reset_index(drop=True)
escogidos=[];intenciones=set()
for i,fila in corpus_voz.iterrows():
    intento=int(fila['intent_class'])
    if intento not in intenciones:
        escogidos.append((i,fila));intenciones.add(intento)
    if len(escogidos)==10:break
if len(escogidos)<10:raise ValueError('No hay 10 categorías distintas en este corpus')
VOZ_RAW=RAW/'voz_real';VOZ_RAW.mkdir(parents=True,exist_ok=True)
audios_reales=[];hashes=set()
for indice,(fila_numero,fila) in enumerate(escogidos,1):
    contenido=fila['audio']
    if not isinstance(contenido,dict) or not isinstance(contenido.get('bytes'),(bytes,bytearray)):
        raise ValueError('Audio no está embebido como bytes en el Parquet; verificar esquema en origen')
    bytes_original=bytes(contenido['bytes'])
    if not bytes_original.startswith(b'RIFF') or bytes_original[8:12]!=b'WAVE':
        raise ValueError('El registro de audio no contiene WAV válido')
    # Los WAV originales de MINDS-14 pueden usar G.711 μ-law; WAV es contenedor, NO códec.
    # Conservar los bytes EXACTOS del editor y generar PCM16 por separado.
    import soundfile as sf
    ruta_original=VOZ_RAW/f'voz_{indice:02}_fuente.wav'
    huella_original=hashlib.sha256(bytes_original).hexdigest()
    if huella_original in hashes:raise ValueError('Se detectó un audio fuente duplicado')
    hashes.add(huella_original)
    if ruta_original.exists() and sha256(ruta_original)!=huella_original:
        raise ValueError('Archivo fuente previamente alterado')
    if not ruta_original.exists():ruta_original.write_bytes(bytes_original)
    with sf.SoundFile(io.BytesIO(bytes_original)) as decodificador:
        frecuencia=decodificador.samplerate
        subtipo=decodificador.subtype
        muestras=decodificador.read(dtype='float32',always_2d=True)
    if not 8000<=frecuencia<=48000 or muestras.shape[0]<frecuencia//2:
        raise ValueError('Audio con duración o tasa de muestreo no compatible')
    mono=np.mean(muestras,axis=1)
    pcm=(np.clip(mono,-1,1)*32767).astype('<i2')
    convertido=io.BytesIO()
    with wave.open(convertido,'wb') as wav_salida:
        wav_salida.setnchannels(1)
        wav_salida.setsampwidth(2)
        wav_salida.setframerate(frecuencia)
        wav_salida.writeframes(pcm.tobytes())
    contenido_pcm=convertido.getvalue()
    ruta=VOZ_RAW/f'voz_{indice:02}.wav'
    huella_pcm=hashlib.sha256(contenido_pcm).hexdigest()
    if ruta.exists() and sha256(ruta)!=huella_pcm:
        raise ValueError('PCM16 anterior no coincide; no se sobrescribe un trabajo modificado')
    if not ruta.exists():ruta.write_bytes(contenido_pcm)
    duracion_real=len(pcm)/frecuencia
    audios_reales.append({'source_record_id':str(fila['path']),'source_row':int(fila_numero),
        'intent_class':int(fila['intent_class']),'transcription_reference':str(fila['transcription']),
        'source_original_path':ruta_original.relative_to(RAIZ).as_posix(),
        'source_original_sha256':huella_original,'source_codec':subtipo,
        'path':ruta.relative_to(RAIZ).as_posix(),'sha256':huella_pcm,
        'duration_seconds':round(duracion_real,3),'license':LICENCIA_VOZ})
assert len(audios_reales)==10 and len(hashes)==10
(VOZ_RAW/'dataset_manifest.json').write_text(json.dumps({
    'dataset':'PolyAI/minds14','subset':'es-ES','license':LICENCIA_VOZ,
    'source_url':ORIGEN_VOZ,'source_parquet_sha256':sha256(ARCHIVO_FUENTE),
    'selection':'one_per_distinct_intent_sorted_by_path','records':audios_reales
},ensure_ascii=False,indent=2),encoding='utf-8')
print('VOCES REALES VERIFICADAS:',len(audios_reales),'· intenciones:',len(intenciones))
display(pd.DataFrame(audios_reales)[['source_record_id','duration_seconds','intent_class','sha256']])

### 7B · ¿Qué dicen las señales de diez personas?

**La pregunta.** Dos audios pueden tener duraciones diferentes e igual contenido temático; y dos frases con intención parecida pueden tener firmas acústicas distintas.

**Ejemplo manual:** para cuatro muestras \`[0, 1, 0, -1]\`, el RMS es \(\sqrt{(0+1+0+1)/4}\), aproximadamente 0,707. La energía depende de la forma de onda, no de si la persona menciona un contrato.

**HAZ ESTO AHORA.** Ejecuta la celda, mira las diez filas y reproduce el audio de la posición elegida (de 1 a 10). Cambia solo \`POSICION_VOZ\` y compara su duración, su RMS y su centroide espectral; contrasta con la transcripción de referencia.

**Contenedor y códec.** En MINDS-14 el archivo fuente puede ser WAV con G.711 μ-law (código 7), no PCM. La transformación real `soundfile → muestras float → wave PCM16` deja dos versiones: fuente inmutable y derivado reproducible.

**Qué significa cada característica:**
- **RMS:** amplitud eficaz; no equivale a volumen percibido ni emoción.
- **ZCR:** frecuencia de cruces por cero por muestra; puede aumentar con componentes de alta frecuencia o ruido, pero no es una clasificación de voz.
- **Centroide espectral:** media de frecuencias ponderada por magnitud espectral; no interpreta palabras.
- **Transcripción de referencia:** texto asociado por el dataset, no predicción de nuestro modelo.

**Cómo se lee:** una fila es una grabación única con procedencia. **Qué nos dice:** variación acústica medible. **Qué NO permite concluir:** la intención del hablante a partir de RMS ni la exactitud de ASR sin ejecutarlo. **Error común:** usar la clave \`article_id\` de noticias para estas voces, que no pertenecen a ellas.

In [ ]:
#@title 7B · Extraer características y escuchar una voz del corpus
def leer_wav_pcm16(ruta):
    with wave.open(str(ruta),'rb') as f:
        tasa=f.getframerate();canales=f.getnchannels();n=f.getnframes()
        if f.getcomptype()!='NONE' or f.getsampwidth()!=2 or not 1<=canales<=2:
            raise ValueError('Se espera WAV PCM de 16 bits mono o estéreo')
        muestras=np.frombuffer(f.readframes(n),dtype='<i2').astype(np.float32)/32768
    if canales>1:muestras=muestras.reshape(-1,canales).mean(axis=1)
    return muestras,tasa
def caracteristicas_voz(ruta):
    muestras,tasa=leer_wav_pcm16(ruta)
    if muestras.size<1024:raise ValueError('Archivo con muy pocas muestras')
    rms=float(np.sqrt(np.mean(muestras.astype(np.float64)**2)))
    cruces=float(np.mean(np.signbit(muestras[1:])!=np.signbit(muestras[:-1])))
    # Primeros 0,5 segundos; Hanning + FFT para el mismo contrato en todas las voces.
    ventana=muestras[:min(len(muestras),int(tasa*.5))]
    amplitudes=np.abs(np.fft.rfft(ventana*np.hanning(len(ventana))))
    hz=np.fft.rfftfreq(len(ventana),d=1/tasa)
    centroide=float(np.sum(hz*amplitudes)/max(1e-12,float(amplitudes.sum())))
    return {'duration_seconds':round(len(muestras)/tasa,3),'sample_rate_hz':int(tasa),
            'rms':round(rms,5),'zero_crossing_rate':round(cruces,5),
            'spectral_centroid_hz':round(centroide,1),'stored_bytes':int(Path(ruta).stat().st_size)}
for item in audios_reales:
    item.update(caracteristicas_voz(RAIZ/item['path']))
tabla_voz=pd.DataFrame(audios_reales)
assert len(tabla_voz)==10 and tabla_voz.sha256.nunique()==10
print('ANÁLISIS ACÚSTICO REAL:',len(tabla_voz),'grabaciones · licencia:',LICENCIA_VOZ)
display(tabla_voz[['source_record_id','intent_class','duration_seconds','rms','zero_crossing_rate','spectral_centroid_hz']])
POSICION_VOZ=1 #@param {type:"integer"}
if type(POSICION_VOZ) is not int or not 1<=POSICION_VOZ<=10:raise ValueError('Selecciona una voz de 1 a 10')
voz_elegida=audios_reales[POSICION_VOZ-1]
senal_escuchada,tasa_escuchada=leer_wav_pcm16(RAIZ/voz_elegida['path'])
print('TRANSCRIPCIÓN HUMANA DE REFERENCIA:',voz_elegida['transcription_reference'])
display(Audio(senal_escuchada,rate=tasa_escuchada))
plt.figure(figsize=(9,2));plt.plot(np.arange(len(senal_escuchada))/tasa_escuchada,senal_escuchada,lw=.65)
plt.title('Forma de onda · voz real '+str(POSICION_VOZ));plt.xlabel('Tiempo (s)');plt.ylabel('Amplitud');plt.show()

### Profundización opcional: transcribir una de las diez voces reales

La señal no equivale a texto. Ahora tenemos **diez audios reales de español** con transcripciones de referencia del corpus MINDS-14. Puedes elegir uno para ejecutar **Whisper tiny** mediante \`faster-whisper\`, sin pagar una API, y calcular WER: la proporción de errores de palabra frente a una referencia.

**HAZ ESTO AHORA (opcional).** Termina primero la preparación de diez audios y selecciona \`POSICION_VOZ\` del 1 al 10. Activa \`USAR_ASR\` solamente si quieres descargar y ejecutar el modelo gratuito en Colab. Puedes elegir el corpus o tu archivo propio autorizado. La salida muestra texto, segmentación y errores respecto a la referencia apropiada.

**Cómo se lee:** WER 0 es coincidencia exacta tras normalización; un número mayor describe errores de transcripción, no veracidad. **Qué nos dice:** desempeño de ese ejemplo de voz. **Qué NO permite concluir:** precisión de todo el corpus, calidad sobre audios de noticias u otra lengua. **Error frecuente:** comparar un audio de voz distinto contra la frase ficticia del ejemplo.

**Si falla:** desactiva \`USAR_ASR\`. El análisis acústico de diez voces y la carga del catálogo siguen siendo obligatorios, pero no dependen de descargar modelos.

In [ ]:
#@title ASR opcional: voz REAL del corpus o grabación propia autorizada
USAR_ASR=False #@param {type:"boolean"}
TIPO_ASR='corpus' #@param ['corpus','propio']
if USAR_ASR:
    if TIPO_ASR=='corpus':
        if not 1<=POSICION_VOZ<=len(audios_reales):raise ValueError('Selecciona primero una voz real')
        fila=audios_reales[POSICION_VOZ-1]
        ruta=RAIZ/fila['path']
        referencia_humana=fila['transcription_reference']
    elif TIPO_ASR=='propio':
        if not USAR_WAV_PROPIO or not AUTORIZADO_PARA_ANALISIS:
            raise ValueError('Habilita un WAV propio con autorización antes del ASR')
        ruta=Path(RUTA_WAV_PROPIO)
        referencia_humana='el contrato tiene tres anexos'
    else:
        raise ValueError('Tipo de fuente de ASR no permitido')
    import importlib.util, subprocess, sys, re
    if importlib.util.find_spec('faster_whisper') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','faster-whisper>=1.0,<2'])
    # Compatibilidad probada: faster-whisper 1.2.1 usa av.open(metadata_errors=...),
    # eliminado en PyAV moderno. Evitar instalar una versión que rompe el decodificador.
    from importlib.metadata import version, PackageNotFoundError
    try:
        pyav_version=version('av')
    except PackageNotFoundError:
        pyav_version=None
    if pyav_version is None or int(pyav_version.split('.')[0])>=18:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','av>=16,<18'])
    from faster_whisper import WhisperModel
    modelo=WhisperModel('tiny',device='cpu',compute_type='int8')
    segmentos,info=modelo.transcribe(str(ruta),language='es',beam_size=2,vad_filter=True)
    segmentos=list(segmentos)
    prediccion=' '.join(s.text.strip() for s in segmentos).strip()
    palabras_ref=re.findall(r'\w+',referencia_humana.casefold(),re.UNICODE)
    palabras_pred=re.findall(r'\w+',prediccion.casefold(),re.UNICODE)
    wer_ejemplo=distancia_palabras(palabras_ref,palabras_pred)/len(palabras_ref) if palabras_ref else None
    print('ASR:',TIPO_ASR,'· archivo:',ruta.name,'· referencia:',referencia_humana)
    print('Predicción:',prediccion,'· WER del ejemplo:',wer_ejemplo)
    print('No es una nota ni un indicador del rendimiento general; guarda el audio y la fuente.')
else:
    print('ASR opcional desactivado; el procesamiento acústico real de diez voces sigue activo.')

### 7C · Lo que proviene de cada fuente y cómo se relaciona con las imágenes

La sesión anterior de imágenes ya dejó un snapshot de **32 imágenes de 24 noticias** (\`Datos/s10_imagenes_reales/inputs.json\`) y un cuaderno específico (\`10_ETL_Multimedia.ipynb\`). Conservamos ese recurso y **reutilizamos su esquema de procedencia**; las diez voces reales provienen de otro conjunto, con otra licencia y otros identificadores.

**HAZ ESTO AHORA.** Lee el manifiesto de las imágenes desde el repositorio (solo metadatos) y comprueba las cifras antes de unir ambos **inventarios de medios**, no sus significados. NO descargamos fotografías del diario ni las volvemos a procesar sin la autorización pertinente.

**Salida esperada:** 32 referencias de imágenes del snapshot anterior; 10 registros de audio de MINDS-14. Se registra explícitamente si existe alguna relación real entre conjuntos: en este caso **ninguna**.

**Límite:** compartir un catálogo técnico no significa compartir un artículo. Si se recibe autorización para analizar audios del mismo medio, se deberá comprobar cada referencia, permisos y enlaces antes de cruzar por \`article_id\`.

In [ ]:
#@title 7C · Integración de inventarios sin inventar relaciones editoriales
RUTA_IMAGENES_CURSO='https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s10_imagenes_reales/inputs.json'
try:
    with urllib.request.urlopen(urllib.request.Request(RUTA_IMAGENES_CURSO,headers={'User-Agent':'BigData-UCentral-S10/1.0'}),timeout=35) as resp:
        imagenes_existentes=json.load(resp)
except Exception as e:
    raise RuntimeError('No se pudieron recuperar los metadatos del snapshot de imágenes; revisa la conexión de Colab') from e
if len(imagenes_existentes)!=32:raise ValueError('El snapshot de imágenes cambió: hay que revisar su contrato')
articulos_imagenes={str(x['article_id']) for x in imagenes_existentes}
if len(articulos_imagenes)!=24:raise ValueError('Cambiaron los article_id del inventario anterior')
registro_fuentes={'sources':[
    {'dataset':'S10 snapshot editorial previo','modality':'image','media_items':32,
     'distinct_articles':24,'id_namespace':'article_id','content_analysis':'sin nueva extracción'},
    {'dataset':'PolyAI/minds14 es-ES','modality':'audio','media_items':10,
     'distinct_articles':0,'id_namespace':'source_record_id','license':'CC-BY-4.0'}],
    'cross_dataset_article_matches':0,'join_policy':'no_unir_fuentes_distintas_por_article_id'}
(OUT/'source_registry.json').write_text(json.dumps(registro_fuentes,ensure_ascii=False,indent=2),encoding='utf-8')
print('INTEGRACIÓN DE INVENTARIOS: 32 referencias de imágenes (24 noticias) + 10 WAV reales autorizados')
print('RELACIONES EDITORIALES ENTRE DATASETS:',registro_fuentes['cross_dataset_article_matches'])

## 8 · Cargar las dos modalidades en un catálogo verificable

La carga (**L**) no consiste en guardar un archivo y olvidarlo. Una fila describe el activo, su modalidad, la procedencia, su huella, sus características y la ruta que permite reabrirlo.

**HAZ ESTO AHORA.** Escribe JSONL y Parquet, vuelve a leer ambos, y contrasta con la misma tabla. El catálogo incluye 2 derivaciones de una imagen de prueba, 1 señal sintética y 10 grabaciones humanas autorizadas. Las voces usan \`source_record_id\`, no \`article_id\`: **no** pertenecen a las noticias de El Tiempo.

In [ ]:
# L — esquema común; dos perfiles de imagen y un audio original del curso.
ARTICULO_DEMO='curso-demo-s10' # No es un artículo de un medio
rows=[]
for lado in (160,320):
    ruta=DER/f'documento_{lado}.png';features=imagen_caracteristicas(ruta)
    rows.append({'article_id':ARTICULO_DEMO,'modality':'image','source_kind':'generated_for_course',
       'asset_id':sha256(imagen_original),'original_path':str(imagen_original.relative_to(RAIZ)),
       'derived_path':str(ruta.relative_to(RAIZ)),'sha256':sha256(ruta),
       'profile':f'vista{lado}','feature_version':'s10-v1',
       'metric_name':'mean_luminance','metric_value':features['mean_luminance'],
       'width':features['width'],'height':features['height'],'duration_seconds':None,'sample_rate_hz':None})
rows.append({'article_id':ARTICULO_DEMO,'modality':'audio','source_kind':'generated_for_course',
   'asset_id':sha256(audio_original),'original_path':str(audio_original.relative_to(RAIZ)),
   'derived_path':str(audio_original.relative_to(RAIZ)),'sha256':sha256(audio_original),
   'profile':'original_pcm16','feature_version':'s10-v1',
   'metric_name':'global_rms','metric_value':round(float(np.sqrt(np.mean(senal**2))),5),
   'width':None,'height':None,'duration_seconds':len(senal)/SR,'sample_rate_hz':SR})
# Integración de los DIEZ WAV reales de voz humana: sin falsa pertenencia a noticias.
for item in audios_reales:
    rows.append({'article_id':None,'source_record_id':item['source_record_id'],
       'collection_id':'PolyAI/minds14-es-ES','modality':'audio',
       'source_kind':'recorded_human_speech_cc_by_4_0','license':item['license'],
       'asset_id':item['source_original_sha256'],'original_path':item['source_original_path'],
       'source_codec':item['source_codec'],'source_original_sha256':item['source_original_sha256'],
       'derived_path':item['path'],'sha256':item['sha256'],
       'profile':'voz_real_pcm16','feature_version':'s10-v1',
       'metric_name':'rms','metric_value':item['rms'],
       'width':None,'height':None,'duration_seconds':item['duration_seconds'],
       'sample_rate_hz':item['sample_rate_hz'],'zero_crossing_rate':item['zero_crossing_rate'],
       'spectral_centroid_hz':item['spectral_centroid_hz'],
       'intent_class':item['intent_class'],
       'transcription_reference':item['transcription_reference']})
catalogo=pd.DataFrame(rows)
catalogo.to_json(OUT/'assets.jsonl',orient='records',lines=True,force_ascii=False)
catalogo.to_parquet(OUT/'assets.parquet',index=False)
reabierto=pd.read_parquet(OUT/'assets.parquet')
jsonl=[json.loads(line) for line in (OUT/'assets.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
assert json.loads(reabierto.to_json(orient='records',force_ascii=False))==jsonl, 'JSONL/Parquet no coinciden'
assert reabierto.groupby('modality').size().to_dict()=={'audio':11,'image':2}
assert len(reabierto[reabierto.source_kind.eq('recorded_human_speech_cc_by_4_0')])==10
display(reabierto[['article_id','modality','profile','metric_name','metric_value','sha256']])
print('L · CATÁLOGO REABIERTO:',len(reabierto),'filas')

## 9 · Tu pregunta, sobre tu resultado

Elige qué modalidad consultar. **No redactes una justificación:** toma una decisión verificable con tu propia salida y contrasta el archivo que se abre.

Si necesitas recuperar el ejemplo, vuelve a elegir \`image\`. No confundas las tres filas del catálogo con tres fuentes independientes: son dos derivaciones de imagen y un audio.

In [ ]:
# Ejercicio de consulta: cambiar SOLO la modalidad.
MODALIDAD='image' #@param ['image','audio']
if MODALIDAD not in ('image','audio'):raise ValueError('Elige una modalidad del catálogo')
resultado=reabierto[reabierto.modality.eq(MODALIDAD)]
assert len(resultado)>0
display(resultado[['modality','profile','metric_name','metric_value','derived_path']])
for _,r in resultado.head(3).iterrows():
    p=RAIZ/r.derived_path
    assert sha256(p)==r.sha256
    if r.modality=='image':display(Image.open(p))
    else:
        with wave.open(str(p),'rb') as wf:
            a=np.frombuffer(wf.readframes(wf.getnframes()),dtype='<i2').astype(np.float32)/32768
            display(Audio(a,rate=wf.getframerate()))
print('CONSULTA PERSONAL:',MODALIDAD,'· filas:',len(resultado),'· muestras reproducidas:',min(3,len(resultado)))

**Cómo se lee:** la consulta selecciona los archivos por modalidad y verifica cada huella antes de abrirlos. **Qué nos dice:** es un catálogo utilizable, no una colección de rutas rotas. **Qué NO permite concluir:** asociación semántica con noticias reales, calidad de OCR o ASR, ni desempeño de un clúster. **Error común:** confundir integridad de bytes con veracidad del contenido.

## 10 · ¿Dónde aparece Big Data?

Un ejemplo pequeño nos permite inspeccionar cada transformación; el diseño debe contemplar muchos activos, pero **no vamos a inventar una prueba de rendimiento distribuido**. Estimar memoria potencial permite decidir qué guardar como binario y qué guardar como características.

Para un millón de imágenes de las mismas dimensiones, compáralas como RGB sin comprimir y como PNG derivado real. Para audio PCM, relaciona segundos, muestras, bytes por muestra y canales. Estas son proyecciones hipotéticas basadas en un ejemplo, **no estadísticas sobre noticias reales**.

In [ ]:
# Proyección matemática, no benchmark ni descarga masiva.
N_ARCHIVOS=1000000
medidas=imagen_caracteristicas(imagen_original)
rgb_bytes=medidas['width']*medidas['height']*3
imagen_bytes=imagen_original.stat().st_size
audio_pcm_bytes=len(senal)*2
print('UN ORIGINAL · imagen sin comprimir:',rgb_bytes,'bytes; PNG en disco:',imagen_bytes,'bytes')
print('UN ORIGINAL · audio WAV PCM16 (sin cabecera):',audio_pcm_bytes,'bytes')
print('1 MILLÓN HIPOTÉTICO de imágenes RGB:',round(rgb_bytes*N_ARCHIVOS/1e9,2),'GB decimales')
print('1 MILLÓN HIPOTÉTICO de audios de',duracion,'s:',round(audio_pcm_bytes*N_ARCHIVOS/1e9,2),'GB decimales')
print('NO SE MIDIERON red, concurrencia, latencia ni almacenamiento distribuido.')
print('DECISIÓN: conservar originales y un índice pequeño de características evita decodificar todo en cada consulta.')

## 11 · Tu decisión: elige y justifica mediante evidencia

Antes de ejecutar la siguiente celda, revisa tus resultados reales y **escoge un valor distinto de 0 en cada selector** de Colab (pulsa «Mostrar código» si necesitas verlos). Las tres selecciones son:

| Selector | Opciones |
|---|---|
| \`PERFIL_IMAGEN\` | **1:** perfil de 160 px, menor resolución; **2:** 320 px, más detalle |
| \`REGION_AUDIO\` | **1:** primer tono, **2:** silencio, **3:** segundo tono |
| \`LIMITE\` | **1:** histograma no hace OCR, **2:** espectro no transcribe, **3:** una muestra pequeña no representa todo el corpus |

**HAZ ESTO AHORA.** Selecciona tres valores no nulos, ejecuta la celda y verifica \`complete: True\`. Explica oralmente qué observaste y cuál alternativa descartaste. No uses una captura del resultado de otra persona: cambia al menos una selección basándote en tus propias observaciones.

**Cómo se lee:** \`decision.json\` registra las opciones elegidas. **Qué nos dice:** que completaste una decisión estructurada. **Qué NO permite concluir:** que las hipótesis semánticas sean verdaderas. **Qué error común:** pensar que una decisión sin contrastar evidencia demuestra comprensión.

In [ ]:
# Selecciones estructuradas, con 0 = pendiente.
PERFIL_IMAGEN=0 #@param [0,1,2] {type:"raw"}
REGION_AUDIO=0 #@param [0,1,2,3] {type:"raw"}
LIMITE=0 #@param [0,1,2,3] {type:"raw"}
for v,validos in [(PERFIL_IMAGEN,(0,1,2)),(REGION_AUDIO,(0,1,2,3)),(LIMITE,(0,1,2,3))]:
    if type(v) is not int or v not in validos:raise ValueError('Código de decisión inválido')
decision={'profile':PERFIL_IMAGEN,'audio_region':REGION_AUDIO,'limit':LIMITE,
          'complete':bool(PERFIL_IMAGEN and REGION_AUDIO and LIMITE),'graded':False}
(CHECKS/'decision.json').write_text(json.dumps(decision,indent=2,ensure_ascii=False),encoding='utf-8')
print('DECISIÓN INDIVIDUAL:',decision)
if not decision['complete']:print('PENDIENTE: selecciona tres códigos distintos de cero para completar tu reflexión.')

In [ ]:
# Manifiesto técnico: huellas y relectura del catálogo/archivos.
from datetime import datetime,timezone
def comprobar(paquete):
    p=Path(paquete);m=json.loads((p/'manifest_s10.json').read_text(encoding='utf-8'))
    errors=[]
    for ruta,esperado in m['files'].items():
        a=p/ruta
        if not a.is_file() or sha256(a)!=esperado:errors.append('hash:'+ruta)
    tabla=pd.read_parquet(p/'outputs/assets.parquet')
    if len(tabla)!=m['rows']:errors.append('rows')
    for _,fila in tabla.iterrows():
        ruta=p/fila['derived_path']
        if not ruta.is_file() or sha256(ruta)!=fila['sha256']:errors.append('asset:'+str(ruta))
    return {'ok':not errors,'errors':errors,'rows':len(tabla)}
filelist=[p for p in RAW.rglob('*') if p.is_file()]+[*DER.glob('*'),OUT/'assets.jsonl',OUT/'assets.parquet',OUT/'source_registry.json',CHECKS/'decision.json']
manifest={'schema':'bigdata-s10-multimodal-v1','created_at_utc':datetime.now(timezone.utc).isoformat(),
          'source_kind':'mixed_synthetic_and_licensed','article_id':'curso-demo-s10',
          'rows':len(reabierto),
          'files':{p.relative_to(RAIZ).as_posix():sha256(p) for p in filelist}}
(RAIZ/'manifest_s10.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
informe=comprobar(RAIZ)
assert informe['ok'],informe
print('PAQUETE VERIFICADO:',informe,'· carpeta:',RAIZ)

## 12 · Autocomprobación: ¿de verdad ejecutaste cada etapa?

El siguiente control **lee tus archivos y variables de ESTA ejecución**, no los de otro estudiante. Revisa imágenes, WAV originales y decodificados, diez claves de voz únicas, transcripciones, características, metadatos del snapshot de 32 imágenes/24 artículos, 13 filas de Parquet y manifiesto. También verifica que hayas completado tus tres decisiones.

**HAZ ESTO AHORA.** Ejecuta la celda. Si todo fue realizado, verás **\`AUTOCOMPROBACIÓN: LISTO\`** y el detalle de cada prueba. Si aparece **\`INCOMPLETO\`**, vuelve a la etapa indicada; por ejemplo, una decisión con 0 es pendiente. La autocomprobación **no evalúa la corrección semántica de tus decisiones**.

**Cómo se lee:** cada indicador es verdadero o falso. **Qué nos dice:** que existe la evidencia técnica necesaria para continuar. **Qué NO permite concluir:** que aprendiste o que las diez grabaciones representan todas las voces. **Qué error común:** marcar manualmente una lista sin verificar archivos.

In [ ]:
#@title 12 · Comprobar el trabajo realizado y diagnosticar pendientes
from collections import OrderedDict
controles=OrderedDict()
controles['01 · Biblioteca y carpeta listas']=RAIZ.is_dir() and RAW.is_dir() and OUT.is_dir()
controles['02 · Imagen de origen y dos tamaños']=all((RAIZ/p).is_file() for p in [
    'raw/documento_demo.png','derived/documento_160.png','derived/documento_320.png'])
controles['03 · Señal de ejemplo']= (RAW/'senal_demo.wav').is_file()
controles['04 · Diez grabaciones reales y diez PCM16']=len(audios_reales)==10 and len({
    x['source_record_id'] for x in audios_reales})==10 and all(
    (RAIZ/x['path']).is_file() and (RAIZ/x['source_original_path']).is_file()
    and sha256(RAIZ/x['path'])==x['sha256']
    and sha256(RAIZ/x['source_original_path'])==x['source_original_sha256']
    for x in audios_reales)
controles['05 · Metadatos y voces con procedencia']=all(
    x.get('transcription_reference') and x.get('license')=='CC-BY-4.0'
    and x.get('sample_rate_hz') and x.get('duration_seconds')>0 for x in audios_reales)
controles['06 · Inventario anterior sin falsas uniones']=len(imagenes_existentes)==32 and len(
    articulos_imagenes)==24 and registro_fuentes['cross_dataset_article_matches']==0
controles['07 · Catálogo 2 imágenes + 11 audios']=len(reabierto)==13 and (
    reabierto.groupby('modality').size().to_dict()=={'audio':11,'image':2})
controles['08 · Manifiesto y huellas']=comprobar(RAIZ)['ok']
controles['09 · Decisión propia completada']=decision.get('complete',False)
for titulo,ok in controles.items():
    print(('✓ ' if ok else '✗ ')+titulo+(': OK' if ok else ': PENDIENTE'))
autocomprobacion_lista=all(controles.values())
print('AUTOCOMPROBACIÓN:', 'LISTO' if autocomprobacion_lista else 'INCOMPLETO',
      f'({sum(controles.values())}/{len(controles)})')
if not autocomprobacion_lista:
    print('Corrige las filas pendientes y vuelve a ejecutar la celda; no entregues el paquete todavía.')

## 13 · Conserva tus evidencias antes de cerrar Colab

**ATENCIÓN:** el contenido de \`/content/\` desaparece cuando se cierra o reinicia el runtime. **Guardar una copia del cuaderno NO basta**. Puedes conservar el paquete de dos maneras:

**Opción A (sin Drive; recomendada para empezar):** ejecuta la celda **Descargar ZIP**. Primero debe mostrar \`AUTOCOMPROBACIÓN: LISTO\`. Activa \`DESCARGAR_ZIP\` para guardar \`s10_multimodal_evidencia.zip\` en Descargas. Descomprímelo y comprueba \`manifest_s10.json\`, \`outputs/assets.parquet\` y \`raw/voz_real\`.

**Opción B (Drive personal):** ejecuta la celda **Guardar en Google Drive**. Cambia \`CONSERVAR_EN_DRIVE\` a verdadero, pulsa ▶ y acepta los permisos de Google Colab. Encontrarás la carpeta en **Mi unidad → BigData2026 → S10 → multimodal**. Verifica que se imprimió \`COPIA REABIERTA: True\`.

**HAZ ESTO AHORA.** Elige al menos una opción. Antes de cerrar la pestaña, comprueba que tienes el ZIP en tu equipo o la carpeta en tu Drive. **No compartas capturas con datos sensibles ni publiques el paquete en un repositorio público sin comprobar las licencias.**

In [ ]:
#@title 13A · Descargar paquete verificable desde Google Colab
DESCARGAR_ZIP=False #@param {type:"boolean"}
if DESCARGAR_ZIP:
    if not autocomprobacion_lista:raise RuntimeError('Primero completa los nueve controles; repite la autocomprobación')
    import zipfile
    destino_zip=RAIZ.parent/'s10_multimodal_evidencia.zip'
    with zipfile.ZipFile(destino_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5) as paquete:
        for archivo in sorted(p for p in RAIZ.rglob('*') if p.is_file()):
            # El Parquet caché solo es una entrada transitoria: los diez WAV verificables sí se incluyen.
            if archivo==ARCHIVO_FUENTE:continue
            paquete.write(archivo,arcname=archivo.relative_to(RAIZ).as_posix())
    with zipfile.ZipFile(destino_zip) as paquete:
        assert 'manifest_s10.json' in paquete.namelist()
        assert 'outputs/assets.parquet' in paquete.namelist()
        assert sum(n.startswith('raw/voz_real/voz_') and n.endswith('.wav') for n in paquete.namelist())==20
        print('ZIP VERIFICADO:',destino_zip.name,'· archivos:',len(paquete.namelist()))
    try:
        from google.colab import files
    except ImportError:
        print('Abre el ZIP desde:',destino_zip)
    else:
        files.download(str(destino_zip))
else:
    print('ZIP desactivado: activa DESCARGAR_ZIP para conservar el trabajo en tu equipo.')

In [ ]:
#@title Opcional: conservar la carpeta íntegra en TU Google Drive (alternativa a descargar un ZIP)
CONSERVAR_EN_DRIVE=False #@param {type:"boolean"}
if CONSERVAR_EN_DRIVE:
    try:
        from google.colab import drive
    except ImportError:
        raise RuntimeError('Esta opción requiere una sesión de Google Colab autenticada')
    drive.mount('/content/drive')
    destino=Path('/content/drive/MyDrive/BigData2026/S10/multimodal')
    destino.parent.mkdir(parents=True,exist_ok=True)
    shutil.copytree(RAIZ,destino,dirs_exist_ok=True)
    verificado=comprobar(destino)
    assert verificado['ok'],verificado
    print('CONSERVADO EN DRIVE:',destino,'· COPIA REABIERTA:',verificado['ok'])
else:
    print('CONSERVACIÓN LOCAL:',RAIZ,'· usa el ZIP descargable o activa la copia a Drive ANTES de cerrar Colab')

## Para llevar · hoja de consulta autónoma

**Producto comprobable:** 10 WAV humanos + señal sintética + 2 imágenes derivadas, catálogo 13 filas, manifiesto, transcripciones de referencia y metadatos de procedencia. La unión con el inventario de 32 imágenes anteriores es a nivel de fuentes: no existe unión semántica entre los dos datasets.

| Necesidad | Operación |
|---|---|
| Conservar original | \`shutil.copyfile()\` / \`Image.save()\` / \`wave.open()\` + SHA-256 |
| Leer píxeles | \`PIL.Image.open()\` → \`np.asarray()\` |
| Reducir imagen | \`thumbnail((lado,lado))\` |
| Medir sonido | \`np.sqrt(np.mean(x**2))\` |
| Frecuencias | \`np.fft.rfft()\` y \`np.fft.rfftfreq()\` |
| Consultar catálogo | \`pd.read_parquet()\` + filtro por modalidad |
| Reabrir y verificar | \`sha256(path)\` + manifiesto |
| Transcribir voz | Modelo ASR autorizado, texto de referencia y evaluación (extensión) |

**Puente a S11:** conserva \`work_s10_multimodal/\`, porque ya tiene el contrato de un catálogo multimedia. La siguiente sesión puede añadir más registros de audio, imágenes reales con permisos, otros formatos y una ejecución por lotes con métricas de escala.

**Checklist autónomo:** ejecutaste la celda de instalación; comparaste dos tamaños; mediste tonos y silencios; procesaste diez WAV distintos; escuchaste dos voces; distinguiste 32 imágenes/24 noticias de las voces; consultaste el catálogo; elegiste tres códigos no nulos; ejecutaste la autocomprobación; descargaste un ZIP o guardaste en Drive. Abre el ZIP o la carpeta copiada y comprueba que contengan `manifest_s10.json` y `outputs/assets.parquet`.

**Idea final:** E obtiene y acredita el origen; T produce características verificables; L almacena una representación que otro proceso puede reabrir. Una característica técnica no equivale a interpretación humana.